# Analyse Exploratoire Enrichie sans Dependances Admin
## Version simplifiee avec Altair et Pandas

## 1. Chargement des Donnees

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Charger le fichier de donnees
df = pd.read_csv("default_of_credit_card_clients.csv")

print("Affichage des premieres lignes du dataset:")
print(df.head())
print(f"\nFormes: {df.shape}")

Affichage des premieres lignes du dataset:
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE   AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   3    90000.0    2          2         2  34.0      0      0      0      0   
1   4    50000.0    2          2         1  37.0      0      0      0      0   
2   5    50000.0    1          2         1  57.0     -1      0     -1      0   
3   6    50000.0    1          1         2  37.0      0      0      0      0   
4   7        NaN    1          1         2  29.0      0      0      0      0   

   ...  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  PAY_AMT4  PAY_AMT5  PAY_AMT6  \
0  ...      15549    1518.0      1500      1000      1000      1000      5000   
1  ...      29547    2000.0      2019      1200      1100      1069      1000   
2  ...      19131    2000.0     36681     10000      9000       689       679   
3  ...      20024    2500.0      1815       657      1000      1000       800   
4  ...     473944   55000.0     40000     38000     20239     13750    

## 2. Informations Generales sur le Dataset

In [ ]:
print(f"Nombre de lignes: {df.shape[0]}")
print(f"Nombre de colonnes: {df.shape[1]}")
print(f"Memoire utilisee: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nColonnes: {list(df.columns)}")

## 3. Types de Donnees et Informations Detaillees

In [ ]:
# Informations detaillees sur les colonnes
info_dict = {
    'Colonne': df.columns,
    'Type': df.dtypes,
    'Non_Null_Count': df.count(),
    'Null_Count': df.isnull().sum(),
    'Unique_Values': df.nunique()
}

info_df = pd.DataFrame(info_dict)
print(info_df.to_string())

## 4. Analyse des Valeurs Manquantes

In [ ]:
# Compter les valeurs manquantes
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Colonne': missing_count.index,
    'Nombre_Manquant': missing_count.values,
    'Pourcentage': missing_percent.values
}).sort_values('Nombre_Manquant', ascending=False)

missing_df = missing_df[missing_df['Nombre_Manquant'] > 0]

if len(missing_df) > 0:
    print("Colonnes avec valeurs manquantes:")
    print(missing_df.to_string())
else:
    print("Aucune valeur manquante detectee!")

print(f"\nTotal de valeurs manquantes: {df.isnull().sum().sum()}")

## 5. Selection des Colonnes Numeriques et Categoriques

In [ ]:
# Selectionner les colonnes numeriques et categoriques
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Nombre de colonnes numeriques: {len(num_cols)}")
print(f"Nombre de colonnes categoriques: {len(cat_cols)}")
print(f"\nColonnes numeriques:")
for col in num_cols:
    print(f"  - {col}")
print(f"\nColonnes categoriques:")
for col in cat_cols:
    print(f"  - {col}")

## 6. Analyse des Valeurs Aberrantes (Outliers)

In [ ]:
# Calcul des outliers avec la methode IQR
outlier_summary = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    outlier_summary[col] = {
        'Nombre_Outliers': len(outliers),
        'Pourcentage': (len(outliers) / len(df)) * 100,
        'Borne_Inferieure': lower_bound,
        'Borne_Superieure': upper_bound
    }

outlier_df = pd.DataFrame(outlier_summary).T.sort_values('Pourcentage', ascending=False)
print("Resume des outliers par colonne (Top 10):")
print(outlier_df.head(10).to_string())
print(f"\nTotal de donnees aberrantes: {outlier_df['Nombre_Outliers'].sum():,}")

## 7. Statistiques Descriptives

In [ ]:
# Statistiques descriptives completes
stats_df = df[num_cols].describe().T
stats_df['Skewness'] = df[num_cols].skew()
stats_df['Kurtosis'] = df[num_cols].kurtosis()

print("Statistiques descriptives des variables numeriques:")
print(stats_df.round(3).to_string())

## 8. Analyse de la Variable Cible

In [ ]:
# Analyser la variable cible
target_col = 'default_payment_next_month'

if target_col in df.columns:
    print(f"Distribution de {target_col}:")
    print("\nCounts:")
    print(df[target_col].value_counts())
    print("\nPourcentages:")
    print((df[target_col].value_counts(normalize=True) * 100).round(2))
    
    # Calculer le ratio de desequilibre
    counts = df[target_col].value_counts()
    if len(counts) == 2:
        ratio = counts.iloc[0] / counts.iloc[1]
        print(f"\nRatio de desequilibre: 1:{ratio:.1f}")
        print(f"Classe 0 (Non-Defaut): {counts.iloc[0]:,} clients ({counts.iloc[0]/len(df)*100:.1f}%)")
        print(f"Classe 1 (Defaut): {counts.iloc[1]:,} clients ({counts.iloc[1]/len(df)*100:.1f}%)")

## 9. Analyse des Variables Categoriques

In [ ]:
# Variables categoriques communes
categorical_features = ['SEX', 'EDUCATION', 'MARRIAGE']

print("Distribution des variables categoriques:")
for col in categorical_features:
    if col in df.columns:
        print(f"\n{col}:")
        dist = df[col].value_counts().sort_index()
        pct = (df[col].value_counts(normalize=True).sort_index() * 100).round(2)
        summary = pd.DataFrame({'Count': dist, 'Percentage': pct})
        print(summary.to_string())

## 10. Correlations et Multicollinearite

In [ ]:
# Calculer la matrice de correlation
corr_matrix = df[num_cols].corr()

# Trouver les paires de variables fortement correlees
print("Paires de variables avec une correlation absolue > 0.8:")
print("="*70)

strong_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) > 0.8:
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            strong_corr_pairs.append((col1, col2, corr_value))
            print(f"{col1} <-> {col2}: {corr_value:.4f}")

if len(strong_corr_pairs) == 0:
    print("Aucune paire avec correlation absolue > 0.8 trouvee")

print(f"\nTotal de paires fortement correlees: {len(strong_corr_pairs)}")

## 11. Correlation avec la Variable Cible

In [ ]:
if target_col in df.columns:
    # Correlations avec la variable cible
    target_corr = df[num_cols].corr()[target_col].drop(target_col).sort_values(ascending=False)
    
    print("Correlation des variables avec la cible (triees):")
    print(target_corr.round(4).to_string())
    
    print(f"\n\nTop 5 variables les plus correlees:")
    for i, (var, corr_val) in enumerate(target_corr.abs().nlargest(5).items(), 1):
        actual_corr = target_corr[var]
        print(f"{i}. {var}: {actual_corr:.4f}")

## 12. Comparaison des Distributions par Classe

In [ ]:
# Comparer les distributions des variables principales par classe
if target_col in df.columns:
    top_vars = target_corr.abs().nlargest(4).index.tolist()
    
    print("Comparaison des distributions par classe pour les variables principales:")
    print("="*70)
    
    for col in top_vars:
        print(f"\n{col}:")
        for class_label in sorted(df[target_col].unique()):
            data = df[df[target_col] == class_label][col].dropna()
            label = "Defaut" if class_label == 1 else "Non-Defaut"
            print(f"  {label}:")
            print(f"    Mean: {data.mean():.2f}, Median: {data.median():.2f}, Std: {data.std():.2f}")

## 13. Resume et Conclusions

In [ ]:
print("="*70)
print("RESUME DE L'ANALYSE EXPLORATOIRE")
print("="*70)

print(f"\n1. STRUCTURE DU DATASET:")
print(f"   Nombre total d'observations: {df.shape[0]:,}")
print(f"   Nombre de variables: {df.shape[1]}")
print(f"   Variables numeriques: {len(num_cols)}")
print(f"   Variables categoriques: {len(cat_cols)}")

print(f"\n2. VALEURS MANQUANTES:")
missing_total = df.isnull().sum().sum()
print(f"   Total de valeurs manquantes: {missing_total}")
if missing_total == 0:
    print("   Conclusion: Aucune valeur manquante detectee")
else:
    print(f"   Pourcentage: {(missing_total / (df.shape[0] * df.shape[1]) * 100):.2f}%")

print(f"\n3. VALEURS ABERRANTES:")
total_outliers = sum(outlier_summary[col]['Nombre_Outliers'] for col in num_cols)
print(f"   Total de donnees aberrantes: {total_outliers:,}")

if target_col in df.columns:
    print(f"\n4. CLASSE CIBLE:")
    counts = df[target_col].value_counts()
    print(f"   Classe 0 (Non-Defaut): {counts.iloc[0]:,} clients ({counts.iloc[0]/len(df)*100:.1f}%)")
    print(f"   Classe 1 (Defaut): {counts.iloc[1]:,} clients ({counts.iloc[1]/len(df)*100:.1f}%)")
    ratio = counts.iloc[0] / counts.iloc[1]
    print(f"   Desequilibre: Ratio de {ratio:.1f}:1")

print(f"\n5. VARIABLES LES PLUS CORRELEES AVEC LA CIBLE:")
for i, (var, corr_val) in enumerate(target_corr.abs().nlargest(5).items(), 1):
    actual_corr = target_corr[var]
    print(f"   {i}. {var}: {actual_corr:.4f}")

print(f"\n6. MULTICOLLINEARITE:")
print(f"   Nombre de paires avec correlation > 0.8: {len(strong_corr_pairs)}")
if len(strong_corr_pairs) > 0:
    print("   Paires principales:")
    for col1, col2, corr in strong_corr_pairs[:3]:
        print(f"      - {col1} <-> {col2}: {corr:.4f}")
else:
    print("   Pas de probleme majeur de multicollinearite")

print("\n" + "="*70)

## 14. Recommendations pour le Nettoyage des Donnees

In [ ]:
recommendations = """
RECOMMANDATIONS POUR LE NETTOYAGE ET LA PREPARATION:

1. TRAITEMENT DES DONNEES MANQUANTES:
   - Utiliser l'imputation par la mediane pour les variables numeriques
   - Ou l'imputation KNN pour une meilleure preservation des relations
   
2. TRAITEMENT DES VALEURS ABERRANTES:
   - Conserver les outliers (donnees financieres legitimes)
   - Appliquer une standardisation robuste (RobustScaler)
   - Envisager une transformation logarithmique pour les variables tres asymetriques
   
3. GESTION DU DESEQUILIBRE DE CLASSES:
   - Appliquer SMOTE sur l'ensemble d'entraînement uniquement
   - Utiliser class_weight='balanced' dans les modeles de classification
   - Utiliser les metriques appropriees: ROC-AUC et Recall, pas Accuracy seule
   
4. ENCODAGE DES VARIABLES CATEGORIQUES:
   - Appliquer OneHotEncoder avec drop='first' pour eviter la multicollinearite
   
5. SELECTION DE VARIABLES:
   - Utiliser SelectKBest ou mutual_info_classif pour reduire la dimensionalite
   - Tenir compte des correlations fortes identifiees
   
6. NORMALISATION:
   - Appliquer RobustScaler pour les variables numeriques
   - Plus robuste que StandardScaler face aux outliers
"""

print(recommendations)

## 15. Etapes Suivantes

In [ ]:
print("Etapes suivantes pour le pipeline de modelisation:")
print()
steps = [
    "1. Nettoyage et transformation des donnees",
    "2. Selection et ingenierie des caracteristiques",
    "3. Division des donnees train/test",
    "4. Equilibrage des classes (SMOTE)",
    "5. Normalisation/standardisation",
    "6. Entrainement de modeles de classification",
    "7. Evaluation et comparaison des modeles",
    "8. Optimisation des hyperparametres",
    "9. Validation finale et interpretabilite"
]
for step in steps:
    print(step)